# Unit Tests Charts

Charts generated from `CI_TEST_RESULTS.md` section **1. Unit Tests** only.

- Source of truth: checked-in `test_report.md` counts listed in `CI_TEST_RESULTS.md`
- Includes only the 18 main comparison versions

## Setup

In [ ]:
import os
import tempfile
from pathlib import Path

cache_dir = Path(tempfile.gettempdir()) / "matplotlib-cache"
cache_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(cache_dir))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", font_scale=1.25)
matplotlib.rcParams["figure.dpi"] = 150
matplotlib.rcParams["savefig.dpi"] = 300
matplotlib.rcParams["figure.facecolor"] = "white"
matplotlib.rcParams["axes.facecolor"] = "white"

REPORTS_DIR = Path("REPORTS")
if not REPORTS_DIR.exists():
    REPORTS_DIR = Path(".")

PASS_COLOR = "#2f9e44"
FAIL_COLOR = "#e03131"
RATE_COLOR = "#1c7ed6"
GRID_COLOR = "#dee2e6"

## Unit Test Data

In [2]:
unit_tests = pd.DataFrame([
    {"Version": "IMBP01", "Feature": "Inventory Management", "Strategy": "BP", "Tool": "Jest", "Passed": 7, "Failed": 0, "Total": 7},
    {"Version": "IMBP02", "Feature": "Inventory Management", "Strategy": "BP", "Tool": "Jest", "Passed": 5, "Failed": 2, "Total": 7},
    {"Version": "IMCE01", "Feature": "Inventory Management", "Strategy": "CE", "Tool": "Jest", "Passed": 7, "Failed": 0, "Total": 7},
    {"Version": "IMCE02", "Feature": "Inventory Management", "Strategy": "CE", "Tool": "Jest", "Passed": 7, "Failed": 0, "Total": 7},
    {"Version": "IMSD01", "Feature": "Inventory Management", "Strategy": "SD", "Tool": "vitest", "Passed": 7, "Failed": 0, "Total": 7},
    {"Version": "IMSD02", "Feature": "Inventory Management", "Strategy": "SD", "Tool": "vitest", "Passed": 7, "Failed": 0, "Total": 7},
    {"Version": "SCBP01", "Feature": "Shopping Cart", "Strategy": "BP", "Tool": "Jest", "Passed": 5, "Failed": 0, "Total": 5},
    {"Version": "SCBP02", "Feature": "Shopping Cart", "Strategy": "BP", "Tool": "Jest", "Passed": 4, "Failed": 1, "Total": 5},
    {"Version": "SCCE01", "Feature": "Shopping Cart", "Strategy": "CE", "Tool": "Jest", "Passed": 5, "Failed": 0, "Total": 5},
    {"Version": "SCCE02", "Feature": "Shopping Cart", "Strategy": "CE", "Tool": "node:test", "Passed": 0, "Failed": 5, "Total": 5},
    {"Version": "SCSD01", "Feature": "Shopping Cart", "Strategy": "SD", "Tool": "vitest", "Passed": 5, "Failed": 0, "Total": 5},
    {"Version": "SCSD02", "Feature": "Shopping Cart", "Strategy": "SD", "Tool": "vitest", "Passed": 5, "Failed": 0, "Total": 5},
    {"Version": "PDBP01", "Feature": "Promotions & Discounts", "Strategy": "BP", "Tool": "Jest", "Passed": 6, "Failed": 0, "Total": 6},
    {"Version": "PDBP02", "Feature": "Promotions & Discounts", "Strategy": "BP", "Tool": "Jest", "Passed": 1, "Failed": 5, "Total": 6},
    {"Version": "PDCE01", "Feature": "Promotions & Discounts", "Strategy": "CE", "Tool": "Jest", "Passed": 6, "Failed": 0, "Total": 6},
    {"Version": "PDCE02", "Feature": "Promotions & Discounts", "Strategy": "CE", "Tool": "node:test", "Passed": 5, "Failed": 1, "Total": 6},
    {"Version": "PDSD01", "Feature": "Promotions & Discounts", "Strategy": "SD", "Tool": "vitest", "Passed": 6, "Failed": 0, "Total": 6},
    {"Version": "PDSD02", "Feature": "Promotions & Discounts", "Strategy": "SD", "Tool": "Jest", "Passed": 5, "Failed": 1, "Total": 6},
])

feature_order = ["Inventory Management", "Shopping Cart", "Promotions & Discounts"]
strategy_order = ["BP", "CE", "SD"]
version_order = [
    "IMBP01", "IMBP02", "IMCE01", "IMCE02", "IMSD01", "IMSD02",
    "SCBP01", "SCBP02", "SCCE01", "SCCE02", "SCSD01", "SCSD02",
    "PDBP01", "PDBP02", "PDCE01", "PDCE02", "PDSD01", "PDSD02",
]

unit_tests["Pass Rate"] = unit_tests["Passed"] / unit_tests["Total"] * 100
unit_tests["Feature"] = pd.Categorical(unit_tests["Feature"], categories=feature_order, ordered=True)
unit_tests["Strategy"] = pd.Categorical(unit_tests["Strategy"], categories=strategy_order, ordered=True)
unit_tests["Version"] = pd.Categorical(unit_tests["Version"], categories=version_order, ordered=True)
unit_tests = unit_tests.sort_values("Version").reset_index(drop=True)

unit_tests

,Version,Feature,Strategy,Tool,Passed,Failed,Total,Pass Rate
0,IMBP01,Inventory Management,BP,Jest,7,0,7,100.000000
1,IMBP02,Inventory Management,BP,Jest,5,2,7,71.428571
2,IMCE01,Inventory Management,CE,Jest,7,0,7,100.000000
3,IMCE02,Inventory Management,CE,Jest,7,0,7,100.000000
4,IMSD01,Inventory Management,SD,vitest,7,0,7,100.000000
5,IMSD02,Inventory Management,SD,vitest,7,0,7,100.000000
6,SCBP01,Shopping Cart,BP,Jest,5,0,5,100.000000
7,SCBP02,Shopping Cart,BP,Jest,4,1,5,80.000000
8,SCCE01,Shopping Cart,CE,Jest,5,0,5,100.000000
9,SCCE02,Shopping Cart,CE,node:test,0,5,5,0.000000


## Strategy Summary

Strategy summary for the 18 main comparison versions.

In [3]:
strategy_source = unit_tests.copy()

strategy_summary = (
    strategy_source.groupby("Strategy", observed=False)[["Passed", "Failed", "Total"]]
    .sum()
    .reindex(strategy_order)
    .reset_index()
)
strategy_summary["Pass Rate"] = strategy_summary["Passed"] / strategy_summary["Total"] * 100
strategy_summary["Pass Rate Label"] = strategy_summary["Pass Rate"].round().astype(int).astype(str) + "%"

expected = pd.DataFrame([
    {"Strategy": "BP", "Passed": 28, "Failed": 8, "Total": 36, "Pass Rate Label": "78%"},
    {"Strategy": "CE", "Passed": 30, "Failed": 6, "Total": 36, "Pass Rate Label": "83%"},
    {"Strategy": "SD", "Passed": 35, "Failed": 1, "Total": 36, "Pass Rate Label": "97%"},
])

pd.testing.assert_frame_equal(
    strategy_summary[["Strategy", "Passed", "Failed", "Total", "Pass Rate Label"]].reset_index(drop=True),
    expected,
    check_dtype=False,
)

strategy_summary

,Strategy,Passed,Failed,Total,Pass Rate,Pass Rate Label
0,BP,28,8,36,77.777778,78%
1,CE,30,6,36,83.333333,83%
2,SD,35,1,36,97.222222,97%


---
## Chart 1: Unit Test Results by Version

Stacked pass/fail counts for every version listed in the Unit Tests section.

In [4]:
fig, ax = plt.subplots(figsize=(13.5, 6.2))

x = np.arange(len(unit_tests))
ax.bar(x, unit_tests["Passed"], color=PASS_COLOR, label="Passed")
ax.bar(x, unit_tests["Failed"], bottom=unit_tests["Passed"], color=FAIL_COLOR, label="Failed")

for idx, row in unit_tests.iterrows():
    ax.text(idx, row["Total"] + 0.35, f"{int(row['Passed'])}/{int(row['Total'])}", ha="center", va="bottom", fontsize=8.5)

ax.set_title("Unit Test Results by Version", fontsize=18, fontweight="bold", pad=14)
ax.set_ylabel("Number of Tests")
ax.set_xticks(x)
ax.set_xticklabels(unit_tests["Version"].astype(str), rotation=45, ha="right")
ax.set_ylim(0, unit_tests["Total"].max() + 3)
ax.grid(axis="y", color=GRID_COLOR)
ax.legend(loc="upper left", ncol=2, frameon=True)

for pos in [5.5, 12.5]:
    ax.axvline(pos, color="#adb5bd", linewidth=1, linestyle="--")

fig.tight_layout()
fig.savefig(REPORTS_DIR / "chart_unit_tests_by_version.png", bbox_inches="tight")
display(fig)
plt.close(fig)

<Figure size 2025x930 with 1 Axes>

---
## Chart 2: Unit Test Pass Rate by Version

Version-level pass rate, highlighting partial and failing implementations.

In [5]:
fig, ax = plt.subplots(figsize=(13.5, 5.6))

colors = [PASS_COLOR if rate == 100 else ("#f08c00" if rate >= 70 else FAIL_COLOR) for rate in unit_tests["Pass Rate"]]
bars = ax.bar(x, unit_tests["Pass Rate"], color=colors)

for bar, rate in zip(bars, unit_tests["Pass Rate"]):
    ax.text(bar.get_x() + bar.get_width() / 2, rate + 2, f"{rate:.0f}%", ha="center", va="bottom", fontsize=8.5)

ax.axhline(100, color="#495057", linewidth=1, linestyle=":")
ax.set_title("Unit Test Pass Rate by Version", fontsize=18, fontweight="bold", pad=14)
ax.set_ylabel("Pass Rate (%)")
ax.set_xticks(x)
ax.set_xticklabels(unit_tests["Version"].astype(str), rotation=45, ha="right")
ax.set_ylim(0, 112)
ax.grid(axis="y", color=GRID_COLOR)

fig.tight_layout()
fig.savefig(REPORTS_DIR / "chart_unit_test_pass_rate_by_version.png", bbox_inches="tight")
display(fig)
plt.close(fig)

<Figure size 2025x840 with 1 Axes>

---
## Chart 3: Test Summary by Strategy

This chart mirrors `### Test Summary by Strategy` for Unit Tests only.

In [6]:
fig, ax = plt.subplots(figsize=(7.8, 5.8))

sx = np.arange(len(strategy_summary))
ax.bar(sx, strategy_summary["Passed"], color=PASS_COLOR, label="Passed")
ax.bar(sx, strategy_summary["Failed"], bottom=strategy_summary["Passed"], color=FAIL_COLOR, label="Failed")

for idx, row in strategy_summary.iterrows():
    ax.text(idx, row["Total"] + 1.2, f"{int(row['Passed'])}/{int(row['Total'])}\n{row['Pass Rate Label']}", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_title("Unit Test Summary by Strategy", fontsize=18, fontweight="bold", pad=14)
ax.set_ylabel("Number of Tests")
ax.set_xticks(sx)
ax.set_xticklabels(["BP\nBasic Prompting", "CE\nContext Engineering", "SD\nSpec-Driven Dev"])
ax.set_ylim(0, strategy_summary["Total"].max() + 9)
ax.grid(axis="y", color=GRID_COLOR)
ax.legend(loc="upper left", ncol=2, frameon=True)

fig.tight_layout()
fig.savefig(REPORTS_DIR / "chart_unit_test_summary_by_strategy.png", bbox_inches="tight")
display(fig)
plt.close(fig)

<Figure size 1170x870 with 1 Axes>

---
## Chart 4: Pass Rate Heatmap by Feature and Strategy

Average pass rate by feature and strategy.

In [7]:
heatmap_data = (
    strategy_source.groupby(["Feature", "Strategy"], observed=False)[["Passed", "Total"]]
    .sum()
    .assign(**{"Pass Rate": lambda df: df["Passed"] / df["Total"] * 100})
    .reset_index()
    .pivot(index="Feature", columns="Strategy", values="Pass Rate")
    .reindex(index=feature_order, columns=strategy_order)
)

fig, ax = plt.subplots(figsize=(7.8, 4.8))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".0f",
    cmap="RdYlGn",
    vmin=0,
    vmax=100,
    linewidths=1,
    linecolor="white",
    cbar_kws={"label": "Pass Rate (%)"},
    ax=ax,
)
ax.set_title("Unit Test Pass Rate by Feature and Strategy", fontsize=16, fontweight="bold", pad=12)
ax.set_xlabel("Strategy")
ax.set_ylabel("Feature")

fig.tight_layout()
fig.savefig(REPORTS_DIR / "chart_unit_test_pass_rate_heatmap.png", bbox_inches="tight")
display(fig)
plt.close(fig)

<Figure size 1170x720 with 2 Axes>